# P1 — Random Group-Aware Split và các mô hình phân loại

Notebook dùng dataset CICIDS2017 đã xử lý. Cell Validation chọn split P1 hoàn chỉnh hiện hành; khi kernel mới, các cell model tự tìm Validation phù hợp qua resolver. Các ID trong output đã lưu chỉ ghi lại những lần chạy trước. Logistic Regression, Decision Tree và Random Forest đã hoàn tất ở mục 6–14; XGBoost mục 15–17 đã hoàn tất. MLP mục 18 có cảnh báo chưa hội tụ sau 100 vòng. Mục 18.1 khảo sát tối đa 200 vòng: ba cấu hình không có cảnh báo, 128 neuron có Macro-F1 Validation cao nhất. Người dùng chọn 128 neuron/200 vòng, đã fit trên toàn Train ở mục 19 và đánh giá Test ở mục 20.

Test được đọc khi audit split ở mục 4; trong các bước chọn cấu hình và fit model, Test không được dùng. Mỗi model được đánh giá Test ở cell riêng sau khi chọn cấu hình và fit trên toàn Train. Test P1 đã được xem nên không dùng để tinh chỉnh hay gọi là holdout cuối hoàn toàn chưa xem. Artifact của các run cũ đã xóa nên chưa xác minh được sự trùng khớp từng dòng giữa các lần Test lịch sử. Cờ `previously_seen_test` trong file kết quả Logistic được ghi cố định, không tự kiểm tra sự trùng khớp dòng.

Các output dưới đây là kết quả đã lưu. Không cần chạy lại để xem kết quả: tạo lại split hoặc Validation có thể tạo run mới, còn các cell model ngăn ghi đè artifact đã tồn tại.

## Quy ước và giới hạn diễn giải

- P1 chia xác định các nhóm feature từ cùng nguồn dữ liệu vào Train/Test; đây không phải đánh giá khả năng tổng quát hóa sang ngày hoặc họ tấn công chưa thấy.
- Split dựa trên hash của **toàn bộ feature**, không dựa trên nhãn hoặc metadata. Hash 64-bit dùng để gom nhóm và xác định phía Train/Test theo ngưỡng. Một vector feature giống nhau luôn nhận cùng phía; va chạm hash (nếu có) chỉ gom thêm hai vector vào cùng partition.
- Ngưỡng hash được đặt trước nhằm lấy khoảng 30% số dòng vào Test. Đây là holdout theo nhóm, không phải stratified split; audit báo chênh lệch phân bố nhãn. Cách chia này không tạo nhiều phân hoạch khác nhau theo seed.
- Dữ liệu được đọc và ghi theo chunks để hạn chế RAM. Các bảng Train/Test được lưu riêng theo `RUN_ID`; không ghi đè một run cũ.

In [1]:
from pathlib import Path
from datetime import datetime
import gc
import gzip
import json
import os
import shutil
import sys
import uuid

import numpy as np
import pandas as pd
import psutil
from IPython.display import display
DATASET_FILENAME = 'cicids2017_clean.csv.gz'
METADATA_COLUMNS = ['SourceFile', 'CaptureDay', 'OriginalLabel', 'BinaryLabel']
LABEL_ORDER = ('Normal', 'Attack')
LABEL_TO_CODE = {'Normal': 0, 'Attack': 1}

TEST_FRACTION = 0.30
DEFAULT_CHUNK_ROWS = 20_000
SAMPLE_ROWS = 5_000

dataset_candidates = [
    root / 'Dataset' / 'Processed' / DATASET_FILENAME
    for root in (Path.cwd(), *Path.cwd().parents)
]
DATASET_PATH = next((path for path in dataset_candidates if path.is_file()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        'Không tìm thấy Dataset/Processed/cicids2017_clean.csv.gz '
        'từ thư mục notebook hoặc các thư mục cha.'
    )

PROCESSED_DIR = DATASET_PATH.parent
OUTPUT_ROOT = PROCESSED_DIR / 'P1_Random_Group_Aware'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / RUN_ID

print(f'Python: {sys.version.split()[0]}')
print(f'Dataset: {DATASET_PATH}')
print(f'P1 run: {RUN_ID}')
print(f'Test mục tiêu: {TEST_FRACTION:.0%}')

Python: 3.10.6
Dataset: D:\HUST\GR2\NIDS-ML\Dataset\Processed\cicids2017_clean.csv.gz
P1 run: 20261114_024012_45f47887
Test mục tiêu: 30%


## 1. Preflight RAM, dung lượng đĩa và schema

Cell này đọc header và một mẫu nhỏ để xác nhận có đúng 78 feature số cùng bốn metadata. Ước lượng bộ nhớ của một chunk rồi giới hạn kích thước chunk theo RAM hiện còn trống; phép chia và audit sẽ không nạp toàn bộ CSV vào DataFrame. Cell cũng kiểm tra dung lượng đĩa sơ bộ cho các file split. Nếu schema hoặc biên an toàn không đạt, cell dừng với lỗi rõ ràng trước khi đọc toàn bộ dataset.

In [2]:
DATA_COLUMNS = pd.read_csv(DATASET_PATH, compression='gzip', nrows=0).columns.tolist()
missing_metadata = sorted(set(METADATA_COLUMNS) - set(DATA_COLUMNS))
if missing_metadata:
    raise ValueError(f'Thiếu metadata trong dataset: {missing_metadata}')

FEATURE_COLUMNS = [column for column in DATA_COLUMNS if column not in METADATA_COLUMNS]
if len(DATA_COLUMNS) != 82 or len(FEATURE_COLUMNS) != 78:
    raise ValueError(
        f'Schema không đúng dataset đã xác minh: {len(DATA_COLUMNS)} cột tổng, '
        f'{len(FEATURE_COLUMNS)} feature.'
    )
FEATURE_DTYPES = {column: np.float64 for column in FEATURE_COLUMNS}
sample = pd.read_csv(
    DATASET_PATH,
    compression='gzip',
    nrows=SAMPLE_ROWS,
    dtype=FEATURE_DTYPES,
)
if sample.empty:
    raise ValueError('Dataset không có dòng dữ liệu.')
if sample.columns.tolist() != DATA_COLUMNS:
    raise ValueError('Schema của sample không khớp header.')

bytes_per_row = sample.memory_usage(index=True, deep=True).sum() / len(sample)
available_bytes = psutil.virtual_memory().available
chunk_budget_bytes = int(available_bytes * 0.10)
estimated_bytes_per_chunk_row = max(bytes_per_row * 3.0, 1.0)
memory_limited_rows = int(chunk_budget_bytes / estimated_bytes_per_chunk_row)
CHUNK_ROWS = min(DEFAULT_CHUNK_ROWS, memory_limited_rows)
if CHUNK_ROWS < 1_000:
    raise MemoryError(
        'RAM khả dụng không đủ biên an toàn cho chunk tối thiểu 1.000 dòng; '
        'hãy giải phóng RAM rồi chạy lại preflight.'
    )
estimated_chunk_peak = int(bytes_per_row * CHUNK_ROWS * 3.0)
dataset_bytes = DATASET_PATH.stat().st_size
free_disk_bytes = shutil.disk_usage(PROCESSED_DIR).free
estimated_split_disk_bytes = max(int(dataset_bytes * 3.0), 512 * 1024**2)
if free_disk_bytes < estimated_split_disk_bytes:
    raise OSError(
        f'Dung lượng đĩa trống {free_disk_bytes / 1024**3:.2f} GiB thấp hơn '
        f'ước tính cần {estimated_split_disk_bytes / 1024**3:.2f} GiB cho P1.'
    )

del sample
gc.collect()
TEST_CUTOFF = np.uint64(int(TEST_FRACTION * (1 << 64)))
print(f'RAM khả dụng: {available_bytes / 1024**3:.2f} GiB')
print(f'Kích thước chunk: {CHUNK_ROWS:,} dòng; peak chunk ước tính: {estimated_chunk_peak / 1024**2:.1f} MiB')
print(f'Đĩa trống: {free_disk_bytes / 1024**3:.2f} GiB; dự kiến cho split: {estimated_split_disk_bytes / 1024**3:.2f} GiB')
print(f'Schema đạt: {len(FEATURE_COLUMNS)} feature số + {len(METADATA_COLUMNS)} metadata.')

RAM khả dụng: 5.40 GiB
Kích thước chunk: 20,000 dòng; peak chunk ước tính: 52.5 MiB
Đĩa trống: 5.52 GiB; dự kiến cho split: 0.66 GiB
Schema đạt: 78 feature số + 4 metadata.


## 2. Đọc tuần tự và kiểm tra dataset processed

Đọc CSV gzip theo chunks để xác nhận schema, nhãn `Normal`/`Attack`, metadata đầy đủ và feature hữu hạn. Đây là bước kiểm tra đầu vào chứ không thay đổi hay làm sạch lại dữ liệu. Bảng kết quả ghi số dòng theo nguồn và nhãn; nếu có giá trị không hợp lệ, cell dừng trước khi tạo split.

In [3]:
def read_dataset_chunks(path, *, float_precision=None):
    return pd.read_csv(
        path,
        compression='gzip',
        chunksize=CHUNK_ROWS,
        dtype=FEATURE_DTYPES,
        float_precision=float_precision,
    )


def validate_chunk(frame, expected_columns=DATA_COLUMNS):
    if frame.columns.tolist() != expected_columns:
        raise ValueError('Schema chunk không khớp dataset đã xác minh.')
    if frame.empty:
        return
    if not frame['BinaryLabel'].isin(LABEL_ORDER).all():
        invalid = frame.loc[~frame['BinaryLabel'].isin(LABEL_ORDER), 'BinaryLabel'].value_counts(dropna=False)
        raise ValueError(f'BinaryLabel không hợp lệ: {invalid.to_dict()}')
    if frame[METADATA_COLUMNS].isna().any().any():
        raise ValueError('Metadata có giá trị thiếu.')
    if frame[METADATA_COLUMNS].astype('string').apply(lambda column: column.str.strip().eq('')).any().any():
        raise ValueError('Metadata có chuỗi rỗng.')
    feature_values = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float64, copy=False)
    if not np.isfinite(feature_values).all():
        raise ValueError('Feature còn NaN hoặc Infinity.')


dataset_rows = 0
dataset_label_counts = {label: 0 for label in LABEL_ORDER}
dataset_source_label_counts = {}
for chunk in read_dataset_chunks(DATASET_PATH):
    validate_chunk(chunk)
    dataset_rows += len(chunk)
    label_counts = chunk['BinaryLabel'].value_counts()
    for label in LABEL_ORDER:
        dataset_label_counts[label] += int(label_counts.get(label, 0))
    grouped = chunk.groupby(['SourceFile', 'BinaryLabel'], sort=False).size()
    for (source, label), count in grouped.items():
        key = (str(source), str(label))
        dataset_source_label_counts[key] = dataset_source_label_counts.get(key, 0) + int(count)
    del chunk
    gc.collect()

if dataset_rows != sum(dataset_label_counts.values()):
    raise AssertionError('Tổng số dòng không khớp tổng Normal + Attack.')
if any(dataset_label_counts[label] == 0 for label in LABEL_ORDER):
    raise ValueError(f'Dataset phải có cả hai lớp: {dataset_label_counts}')
if dataset_rows != 2_572_640 or dataset_label_counts != {'Normal': 2_146_899, 'Attack': 425_741}:
    raise ValueError(
        'Counts khác dataset đã xác minh (2.572.640 dòng; Normal 2.146.899; '
        'Attack 425.741). Dừng để xác minh lại nguồn, không tiếp tục với split khác.'
    )

dataset_audit = pd.DataFrame(
    [
        {'SourceFile': source, 'BinaryLabel': label, 'Rows': rows}
        for (source, label), rows in sorted(dataset_source_label_counts.items())
    ]
)
display(dataset_audit)
print(f'Tổng: {dataset_rows:,}; Normal: {dataset_label_counts["Normal"]:,}; Attack: {dataset_label_counts["Attack"]:,}')
print('Schema, metadata, nhãn và feature hữu hạn đã được kiểm tra theo chunks.')

,SourceFile,BinaryLabel,Rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Attack,128014
1,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,95068
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Attack,90694
3,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,123083
4,Friday-WorkingHours-Morning.pcap_ISCX.csv,Attack,1948
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Normal,182096
6,Monday-WorkingHours.pcap_ISCX.csv,Normal,502650
7,Thursday-WorkingHours-Afternoon-Infilteration....,Attack,36
8,Thursday-WorkingHours-Afternoon-Infilteration....,Normal,252754
9,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Attack,2143


Tổng: 2,572,640; Normal: 2,146,899; Attack: 425,741
Schema, metadata, nhãn và feature hữu hạn đã được kiểm tra theo chunks.


## 3. Tạo Random Group-Aware Split khoảng 70/30

Mỗi dòng được gán một hash từ toàn bộ 78 feature sau khi đọc thống nhất thành `float64`. Hash được so với ngưỡng 30% đã đặt trước; nhãn, `SourceFile`, `CaptureDay` và `OriginalLabel` không tham gia quyết định partition. Vì các dòng có cùng vector luôn có cùng hash, chúng không bị tách qua Train/Test.

Cell ghi Train và Test theo chunks vào file gzip tạm trong thư mục run riêng; phân bố nhãn sẽ được báo ở bước audit. Khi đọc lại split, dùng `float_precision='round_trip'` để khôi phục giá trị float64 đã ghi.

In [4]:
def feature_group_hashes(frame):
    return pd.util.hash_pandas_object(
        frame.loc[:, FEATURE_COLUMNS],
        index=False,
    ).to_numpy(dtype=np.uint64, copy=False)


if RUN_DIR.exists():
    raise FileExistsError(f'P1 run directory đã tồn tại; không ghi đè: {RUN_DIR}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=False, exist_ok=False)

TRAIN_TEMP_PATH = RUN_DIR / 'train.csv.gz.tmp'
TEST_TEMP_PATH = RUN_DIR / 'test.csv.gz.tmp'
if TRAIN_TEMP_PATH.exists() or TEST_TEMP_PATH.exists():
    raise FileExistsError('Tìm thấy split tạm; không ghi nối vào output cũ.')

train_rows_written = 0
test_rows_written = 0
train_header_written = False
test_header_written = False
with gzip.open(TRAIN_TEMP_PATH, 'wt', encoding='utf-8', newline='') as train_file, gzip.open(
    TEST_TEMP_PATH, 'wt', encoding='utf-8', newline=''
) as test_file:
    for chunk in read_dataset_chunks(DATASET_PATH):
        validate_chunk(chunk)
        hashes = feature_group_hashes(chunk)
        is_test = hashes < TEST_CUTOFF
        test_chunk = chunk.loc[is_test]
        train_chunk = chunk.loc[~is_test]

        if not train_chunk.empty:
            train_chunk.to_csv(train_file, index=False, header=not train_header_written)
            train_header_written = True
            train_rows_written += len(train_chunk)
        if not test_chunk.empty:
            test_chunk.to_csv(test_file, index=False, header=not test_header_written)
            test_header_written = True
            test_rows_written += len(test_chunk)
        del chunk, hashes, is_test, test_chunk, train_chunk
        gc.collect()

if not train_header_written or not test_header_written:
    raise ValueError('Train hoặc Test rỗng; không thể tiếp tục P1.')
if train_rows_written + test_rows_written != dataset_rows:
    raise AssertionError('Số dòng Train + Test không bằng dataset đầu vào.')
print(f'Đã ghi split tạm: Train {train_rows_written:,}; Test {test_rows_written:,}.')
print(f'Tỷ lệ Test thực tế: {test_rows_written / dataset_rows:.3%}; chưa chạy model.')

Đã ghi split tạm: Train 1,800,661; Test 771,979.
Tỷ lệ Test thực tế: 30.007%; chưa chạy model.


## 4. Audit split P1 — điểm dừng trước huấn luyện

Đọc lại cả hai split theo chunks. Cell kiểm tra schema/nhãn/metadata/feature, đối chiếu tổng dòng và số Normal/Attack, đếm feature-group duy nhất, nhóm cùng feature khác nhãn, và giao giữa các group Train/Test. Giao group phải bằng 0. Bảng phân bố nhãn cho biết tỷ lệ Attack của toàn bộ dữ liệu, Train và Test; chênh trên 1 điểm phần trăm được cảnh báo để người dùng xem xét. Audit giữ các mảng hash và nhãn trong RAM; kiểm tra bộ nhớ dưới đây chỉ là ước tính, không bảo đảm mức peak thực tế.

Nếu các kiểm tra đạt, file split tạm được công bố trong thư mục run và audit/manifest được lưu. Đây là điểm kiểm tra trước huấn luyện; split đã lưu bên dưới đã được duyệt trước khi tiếp tục các mục sau.

In [5]:
audit_ram_estimate = dataset_rows * 128 + estimated_chunk_peak
audit_ram_available = psutil.virtual_memory().available
print(f'RAM bổ sung ước tính cho audit: {audit_ram_estimate / 1024**2:.1f} MiB; khả dụng: {audit_ram_available / 1024**2:.1f} MiB.')
if audit_ram_estimate > audit_ram_available * 0.5:
    raise MemoryError('RAM khả dụng không đủ biên an toàn ước tính cho audit; giải phóng RAM rồi chạy lại mục 4.')

def audit_partition(path, partition_name):
    row_count = 0
    label_counts = {label: 0 for label in LABEL_ORDER}
    hash_parts = []
    label_code_parts = []

    for chunk in read_dataset_chunks(path, float_precision='round_trip'):
        validate_chunk(chunk)
        row_count += len(chunk)
        labels = chunk['BinaryLabel'].map(LABEL_TO_CODE)
        if labels.isna().any():
            raise ValueError(f'{partition_name} có nhãn không ánh xạ được.')
        for label in LABEL_ORDER:
            label_counts[label] += int(chunk['BinaryLabel'].eq(label).sum())
        hash_parts.append(feature_group_hashes(chunk))
        label_code_parts.append(labels.to_numpy(dtype=np.uint8, copy=False))
        del chunk, labels
        gc.collect()

    if row_count == 0:
        raise ValueError(f'{partition_name} không có dòng dữ liệu.')
    return {
        'name': partition_name,
        'rows': row_count,
        'label_counts': label_counts,
        'hashes': np.concatenate(hash_parts),
        'label_codes': np.concatenate(label_code_parts),
    }


train_audit = audit_partition(TRAIN_TEMP_PATH, 'Train')
test_audit = audit_partition(TEST_TEMP_PATH, 'Test')
if train_audit['rows'] + test_audit['rows'] != dataset_rows:
    raise AssertionError('Tổng dòng sau split khác dataset đầu vào.')
for label in LABEL_ORDER:
    observed = train_audit['label_counts'][label] + test_audit['label_counts'][label]
    if observed != dataset_label_counts[label]:
        raise AssertionError(f'Số dòng {label} sau split không khớp dataset.')
    if train_audit['label_counts'][label] == 0 or test_audit['label_counts'][label] == 0:
        raise ValueError(f'Không có đủ hai lớp trong cả Train và Test: {label}.')

train_unique_hashes = np.unique(train_audit['hashes'])
test_unique_hashes = np.unique(test_audit['hashes'])
overlap_hashes = np.intersect1d(train_unique_hashes, test_unique_hashes, assume_unique=True)
if overlap_hashes.size:
    raise AssertionError(f'Phát hiện {overlap_hashes.size:,} feature group trong cả Train và Test.')

def count_mixed_label_groups(hashes, label_codes):
    pairs = np.empty(len(hashes), dtype=[('group_hash', np.uint64), ('label_code', np.uint8)])
    pairs['group_hash'] = hashes
    pairs['label_code'] = label_codes
    unique_pairs = np.unique(pairs)
    sorted_hashes = unique_pairs['group_hash']
    return int(np.count_nonzero(sorted_hashes[1:] == sorted_hashes[:-1]))


train_mixed_label_groups = count_mixed_label_groups(train_audit['hashes'], train_audit['label_codes'])
test_mixed_label_groups = count_mixed_label_groups(test_audit['hashes'], test_audit['label_codes'])
conflicting_label_groups = train_mixed_label_groups + test_mixed_label_groups

test_share = test_audit['rows'] / dataset_rows
if not 0.28 <= test_share <= 0.32:
    print(f'CẢNH BÁO: tỷ lệ Test là {test_share:.2%}, lệch khỏi khoảng audit 28–32%; cần xem xét.')

distribution_rows = []
for split_name, row_count, counts in (
    ('Dataset', dataset_rows, dataset_label_counts),
    ('Train', train_audit['rows'], train_audit['label_counts']),
    ('Test', test_audit['rows'], test_audit['label_counts']),
):
    attack_pct = 100.0 * counts['Attack'] / row_count
    overall_attack_pct = 100.0 * dataset_label_counts['Attack'] / dataset_rows
    distribution_rows.append({
        'Split': split_name,
        'Rows': row_count,
        'Normal': counts['Normal'],
        'Attack': counts['Attack'],
        'AttackPct': attack_pct,
        'DifferenceVsDataset_pp': attack_pct - overall_attack_pct,
    })
label_distribution_audit = pd.DataFrame(distribution_rows)
for split_name in ('Train', 'Test'):
    difference_pp = abs(float(label_distribution_audit.loc[label_distribution_audit['Split'].eq(split_name), 'DifferenceVsDataset_pp'].iloc[0]))
    if difference_pp > 1.0:
        print(f'CẢNH BÁO: AttackPct ở {split_name} lệch {difference_pp:.3f} điểm phần trăm so với Dataset; cần xem xét.')

group_audit = pd.DataFrame([
    {'Partition': 'Train', 'Rows': train_audit['rows'], 'UniqueFeatureGroups': len(train_unique_hashes), 'DuplicateFeatureRows': train_audit['rows'] - len(train_unique_hashes)},
    {'Partition': 'Test', 'Rows': test_audit['rows'], 'UniqueFeatureGroups': len(test_unique_hashes), 'DuplicateFeatureRows': test_audit['rows'] - len(test_unique_hashes)},
    {'Partition': 'Cross-partition audit', 'Rows': dataset_rows, 'UniqueFeatureGroups': len(train_unique_hashes) + len(test_unique_hashes), 'DuplicateFeatureRows': dataset_rows - len(train_unique_hashes) - len(test_unique_hashes)},
])
group_audit['SharedFeatureGroupsTrainTest'] = [0, 0, int(overlap_hashes.size)]
group_audit['MixedLabelFeatureGroups'] = [train_mixed_label_groups, test_mixed_label_groups, conflicting_label_groups]

split_counts = {
    'Train': {'rows': train_audit['rows'], **train_audit['label_counts']},
    'Test': {'rows': test_audit['rows'], **test_audit['label_counts']},
}
split_manifest = {
    'run_id': RUN_ID,
    'created_at': datetime.now().astimezone().isoformat(),
    'source_dataset': DATASET_PATH.resolve().relative_to(DATASET_PATH.resolve().parents[2]).as_posix(),
    'source_dataset_bytes': DATASET_PATH.stat().st_size,
    'split_method': 'feature-group hash threshold',
    'test_fraction_target': TEST_FRACTION,
    'test_fraction_observed': test_share,
    'hash_method': 'pandas.util.hash_pandas_object, uint64, all 78 float64 features',
    'split_csv_float_precision': 'round_trip',
    'metadata_columns_excluded_from_model': METADATA_COLUMNS,
    'feature_columns': FEATURE_COLUMNS,
    'split_counts': split_counts,
    'feature_groups_train': int(len(train_unique_hashes)),
    'feature_groups_test': int(len(test_unique_hashes)),
    'shared_feature_groups_train_test': int(overlap_hashes.size),
    'mixed_label_feature_groups': conflicting_label_groups,
}

TRAIN_PATH = RUN_DIR / 'train.csv.gz'
TEST_PATH = RUN_DIR / 'test.csv.gz'
if TRAIN_PATH.exists() or TEST_PATH.exists():
    raise FileExistsError('File split đích đã tồn tại; không ghi đè.')
os.replace(TRAIN_TEMP_PATH, TRAIN_PATH)
os.replace(TEST_TEMP_PATH, TEST_PATH)
label_distribution_audit.to_csv(RUN_DIR / 'label_distribution_audit.csv', index=False, encoding='utf-8')
group_audit.to_csv(RUN_DIR / 'feature_group_audit.csv', index=False, encoding='utf-8')
with (RUN_DIR / 'split_manifest.json').open('w', encoding='utf-8') as manifest_file:
    json.dump(split_manifest, manifest_file, ensure_ascii=False, indent=2)

display(label_distribution_audit)
display(group_audit)
print(f'Feature group giao Train/Test: {overlap_hashes.size:,} (bắt buộc bằng 0).')
print(f'Nhóm cùng feature khác nhãn được giữ: {conflicting_label_groups:,}.')
print(f'Kiểm tra cấu trúc đạt; hãy xem cảnh báo và duyệt audit trước khi huấn luyện: {RUN_DIR}')

RAM bổ sung ước tính cho audit: 366.6 MiB; khả dụng: 6524.6 MiB.


,Split,Rows,Normal,Attack,AttackPct,DifferenceVsDataset_pp
0,Dataset,2572640,2146899,425741,16.548798,0.000000
1,Train,1800661,1502677,297984,16.548590,-0.000208
2,Test,771979,644222,127757,16.549284,0.000486


,Partition,Rows,UniqueFeatureGroups,DuplicateFeatureRows,SharedFeatureGroupsTrainTest,MixedLabelFeatureGroups
0,Train,1800661,1763930,36731,0,497
1,Test,771979,756171,15808,0,200
2,Cross-partition audit,2572640,2520101,52539,0,697


Feature group giao Train/Test: 0 (bắt buộc bằng 0).
Nhóm cùng feature khác nhãn được giữ: 697.
Kiểm tra cấu trúc đạt; hãy xem cảnh báo và duyệt audit trước khi huấn luyện: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887


## Điểm kiểm tra audit P1

Kiểm tra tổng số dòng, tỷ lệ nhãn, tỷ lệ Test và `SharedFeatureGroupsTrainTest = 0`. Các nhóm feature có nhãn mâu thuẫn vẫn được giữ nguyên trong một partition. Không chọn phân hoạch theo kết quả Test. Audit đã được duyệt; mục 5 tạo Validation từ Train của run đã chọn.

## 5. Chia Train nội bộ và Validation

Dùng `RUN_DIR` hiện tại nếu thư mục đó tồn tại trong kernel; nếu không, tự tìm split P1 hoàn chỉnh mới nhất có manifest. Cell chia Train bằng `GroupShuffleSplit` (mục tiêu 20% nhóm feature cho Validation); không đọc nội dung Test. Kết quả được lưu trong thư mục `validation_runs` của split được chọn và audit đã được duyệt trước khi fit model. Chạy lại cell sẽ tạo một Validation run mới.

In [1]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'Notebooks' / 'validation_split.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import validation_split
importlib.reload(validation_split)
if 'RUN_DIR' in globals() and Path(RUN_DIR).is_dir():
    p1_source_run = Path(RUN_DIR)
else:
    p1_source_run = validation_split.latest_p1_split_run_dir(project_root)
print(f"P1 source run: {p1_source_run}")
p1_validation = validation_split.create_internal_validation_split('P1', p1_run_dir=p1_source_run)
print(f"Validation artifacts: {p1_validation['run_dir']}")
display(p1_validation['partition_audit'])
display(p1_validation['day_label_audit'])
display(p1_validation['original_label_audit'])


Validation artifacts: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2


,Partition,Rows,Normal,Attack,AttackRatePercent,UniqueFeatureGroups
0,TrainInternal,1440435,1201975,238460,16.554721,1411144
1,Validation,360226,300702,59524,16.524071,352786


,Partition,CaptureDay,BinaryLabel,Rows
0,TrainInternal,Friday,Attack,123958
1,TrainInternal,Friday,Normal,224213
2,TrainInternal,Monday,Normal,280678
3,TrainInternal,Thursday,Attack,1240
4,TrainInternal,Thursday,Normal,232163
5,TrainInternal,Tuesday,Attack,5140
6,TrainInternal,Tuesday,Normal,231573
7,TrainInternal,Wednesday,Attack,108122
8,TrainInternal,Wednesday,Normal,233348
9,Validation,Friday,Attack,30652


,Partition,OriginalLabel,BinaryLabel,Rows
0,TrainInternal,BENIGN,Normal,1201975
1,TrainInternal,Bot,Attack,1104
2,TrainInternal,DDoS,Attack,71666
3,TrainInternal,DoS GoldenEye,Attack,5744
4,TrainInternal,DoS Hulk,Attack,96445
5,TrainInternal,DoS Slowhttptest,Attack,2879
6,TrainInternal,DoS slowloris,Attack,3046
7,TrainInternal,FTP-Patator,Attack,3385
8,TrainInternal,Heartbleed,Attack,8
9,TrainInternal,Infiltration,Attack,18


## 6. Logistic Regression — so sánh trên Validation

Sau khi duyệt audit, thử `C = 0.1, 1, 10`; các tham số khác giữ như baseline. Mỗi cấu hình fit `StandardScaler` và model trên Train nội bộ, rồi dự đoán Validation. Chọn Macro-F1 cao nhất; nếu bằng nhau chính xác, chọn C nhỏ hơn. Bảng cũng báo Recall, FPR và các giá trị TN/FP/FN/TP; không dùng Test để chọn. Nếu phát sinh `ConvergenceWarning`, cell dừng để xem lại thay vì âm thầm chọn.

In [2]:
RUN_P1_LOGISTIC_TUNING = True  
if not RUN_P1_LOGISTIC_TUNING:
    raise RuntimeError('Chưa duyệt chạy Logistic trên Validation.')
import importlib
from IPython.display import display
import validation_models
importlib.reload(validation_models)
import p1_logistic_workflow
importlib.reload(p1_logistic_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_logistic_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
p1_logistic_workflow.prepare_validation_cache(p1_validation['run_dir'])
logistic_validation_results = p1_logistic_workflow.compare_validation(p1_validation['run_dir'])
display(logistic_validation_results)


Prepared 1,440,435 internal Train rows x 78 features.
Normal: 1,201,975; Attack: 238,460
Cache saved to: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\training_cache
Đang fit và đánh giá Validation: C=0.1
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 6.84 GiB.
Đang fit và đánh giá Validation: C=1.0
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 5.87 GiB.
Đang fit và đánh giá Validation: C=10.0
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 6.28 GiB.
C được chọn theo Macro-F1: 10.0


,C,fit_seconds,n_iter,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,0.1,59.239812,33,360226,297388,3314,24971,34553,0.921480,0.912483,0.580489,0.709573,0.832088,0.011021,0.419511
1,1.0,50.378248,33,360226,297392,3310,24971,34553,0.921491,0.912580,0.580489,0.709602,0.832106,0.011008,0.419511
2,10.0,53.965026,33,360226,297397,3305,24971,34553,0.921505,0.912700,0.580489,0.709638,0.832128,0.010991,0.419511


## 7. Train lại Logistic trên Train gốc

Sau khi xem bảng Validation, fit pipeline mới với C đã chọn trên toàn bộ Train gốc. Scaler cũng được fit lại chỉ trên Train gốc; lưu model và cấu hình riêng trong run Validation. Cell không đọc Test.

In [3]:
RUN_P1_LOGISTIC_FINAL = True
if not RUN_P1_LOGISTIC_FINAL:
    raise RuntimeError('Chưa duyệt train lại Logistic trên Train gốc.')
import importlib
import validation_models
importlib.reload(validation_models)
import p1_logistic_workflow
importlib.reload(p1_logistic_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_logistic_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
logistic_model_path = p1_logistic_workflow.fit_final(p1_validation['run_dir'])
print(logistic_model_path)


Peak RAM ước tính: 2.09 GiB; RAM khả dụng: 4.30 GiB.
Đã lưu pipeline Logistic: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\logistic_selected_pipeline.joblib
D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\logistic_selected_pipeline.joblib


## 8. Đánh giá Logistic trên Test P1

Sau khi fit model được chọn, dự đoán trên Test và xem Recall, FPR, Macro-F1 cùng các giá trị TN/FP/FN/TP. Test không dùng để chọn cấu hình hoặc fit model; cell này chỉ đánh giá pipeline đã fit. Không dùng kết quả Test để chỉnh cấu hình.

In [4]:
RUN_P1_LOGISTIC_TEST = True
if not RUN_P1_LOGISTIC_TEST:
    raise RuntimeError('Chưa duyệt đánh giá Test P1.')
import importlib
from IPython.display import display
import validation_models
importlib.reload(validation_models)
import p1_logistic_workflow
importlib.reload(p1_logistic_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_logistic_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
logistic_test_results = p1_logistic_workflow.evaluate_test(p1_validation['run_dir'])
display(logistic_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,771979,633636,10586,53394,74363,0.917122,0.875384,0.582066,0.699209,0.825575,0.016432,0.417934


## 9. Decision Tree — so sánh trên Validation

Dùng cùng Train nội bộ, Validation và 78 feature của P1; cây không cần `StandardScaler`. So sánh 5 cấu hình Gini với `max_depth` 12/16/20 và `min_samples_leaf` 10/25/50; `max_leaf_nodes=50.000`, `class_weight=None`, seed 42. Chọn Macro-F1 cao nhất; nếu bằng nhau chính xác, ưu tiên cây nông hơn rồi `min_samples_leaf` lớn hơn. Bảng có thêm Recall, FPR, TN/FP/FN/TP, độ sâu thực tế và số lá. Test không được đọc ở bước này. Nếu thiếu state trong kernel, cell tự tìm Validation run phù hợp; cần kiểm tra đúng run trước khi chạy.

In [3]:
RUN_P1_DECISION_TREE_VALIDATION = True
if not RUN_P1_DECISION_TREE_VALIDATION:
    raise RuntimeError('Chưa duyệt chạy Decision Tree trên Validation.')
import importlib
from pathlib import Path
import sys
from IPython.display import display
project_root = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'Notebooks' / 'validation_models.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import validation_models
importlib.reload(validation_models)
import p1_decision_tree_workflow
importlib.reload(p1_decision_tree_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_decision_tree_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
p1_decision_tree_workflow.prepare_validation_cache(p1_validation['run_dir'])
decision_tree_validation_results = p1_decision_tree_workflow.compare_validation(p1_validation['run_dir'])
display(decision_tree_validation_results)


Đã nạp lại Validation run: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2
Using validated training cache: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\training_cache
Đang fit và đánh giá Validation: max_depth=12, min_samples_leaf=10
Peak RAM ước tính: 1.77 GiB; RAM khả dụng: 4.72 GiB.
Đang fit và đánh giá Validation: max_depth=16, min_samples_leaf=10
Peak RAM ước tính: 1.77 GiB; RAM khả dụng: 3.80 GiB.
Đang fit và đánh giá Validation: max_depth=20, min_samples_leaf=10
Peak RAM ước tính: 1.77 GiB; RAM khả dụng: 4.34 GiB.
Đang fit và đánh giá Validation: max_depth=20, min_samples_leaf=25
Peak RAM ước tính: 1.77 GiB; RAM khả dụng: 4.62 GiB.
Đang fit và đánh giá Validation: max_depth=20, min_samples_leaf=50
Peak RAM ước tính: 1.77 GiB; RAM khả dụng: 4.04 GiB.
Cấu hình được chọn theo Macro-F1: {'criterion': 'gini', 'max_leaf_nodes': 

,max_depth,min_samples_leaf,max_leaf_nodes,fit_seconds,observed_depth,leaf_count,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,12,10,50000,78.774926,12,172,360226,300487,215,782,58742,0.997232,0.996353,0.986862,0.991585,0.994964,0.000715,0.013138
1,16,10,50000,96.621346,16,280,360226,300439,263,318,59206,0.998387,0.995578,0.994658,0.995117,0.997076,0.000875,0.005342
2,20,10,50000,111.724516,20,505,360226,300401,301,119,59405,0.998834,0.994959,0.998001,0.996477,0.997889,0.001001,0.001999
3,20,25,50000,124.659086,20,472,360226,300375,327,163,59361,0.998640,0.994522,0.997262,0.995890,0.997537,0.001087,0.002738
4,20,50,50000,142.328968,20,402,360226,300313,389,199,59325,0.998368,0.993486,0.996657,0.995069,0.997045,0.001294,0.003343


## 10. Fit Decision Tree đã chọn trên toàn Train

Sau khi xem và duyệt bảng Validation ở mục 9, fit cấu hình đã chọn trên toàn bộ Train gốc, bao gồm cả phần từng dùng làm Validation; không đọc Test. Model cùng thông tin cấu hình được lưu trong thư mục Validation hiện tại; không ghi đè artifact đã tồn tại.

In [5]:
RUN_P1_DECISION_TREE_FINAL = True
if not RUN_P1_DECISION_TREE_FINAL:
    raise RuntimeError('Chưa duyệt fit Decision Tree trên toàn Train.')
import p1_decision_tree_workflow
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_decision_tree_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
decision_tree_model_path = p1_decision_tree_workflow.fit_final(p1_validation['run_dir'])
print(decision_tree_model_path)


Peak RAM ước tính: 2.08 GiB; RAM khả dụng: 4.87 GiB.
Đã lưu Decision Tree: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\decision_tree_selected_model.joblib
D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\decision_tree_selected_model.joblib


## 11. Đánh giá Decision Tree trên Test P1

Chỉ chạy sau khi đã duyệt kết quả Validation và fit model cuối trên Train. Cell dự đoán Test để báo cáo Accuracy, Precision/Recall Attack, Macro-F1, FPR/FNR và TN/FP/FN/TP; không dùng Test để chọn cấu hình hoặc chỉnh model.

In [7]:
RUN_P1_DECISION_TREE_TEST = True
if not RUN_P1_DECISION_TREE_TEST:
    raise RuntimeError('Chưa duyệt đánh giá Decision Tree trên Test P1.')
import p1_decision_tree_workflow
from IPython.display import display
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_decision_tree_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
decision_tree_test_results = p1_decision_tree_workflow.evaluate_test(p1_validation['run_dir'])
display(decision_tree_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,771979,643562,660,291,127466,0.998768,0.994849,0.997722,0.996283,0.997773,0.001024,0.002278


## 12. Random Forest — so sánh trên Validation

Dùng cùng 78 feature, Train nội bộ và Validation P1; không cần scaler. Thử tuần tự các cặp `(max_depth, min_samples_leaf)`: (12, 20), (16, 20), (20, 20), (20, 10), (20, 50). Mỗi cấu hình có 50 cây, Gini, tối đa 10.000 lá/cây, `max_features='sqrt'`, `bootstrap=True`, `max_samples=0.25`, `class_weight=None`, `random_state=42`, `n_jobs=1`. Mỗi cây lấy có hoàn lại số lượt bằng 25% Train, không phải 25% dòng duy nhất. Chọn Macro-F1 cao nhất; nếu hòa chính xác, ưu tiên depth nhỏ hơn rồi leaf lớn hơn. RAM được ước tính và dừng khi vượt 70% RAM khả dụng; ước tính không bảo đảm peak thực tế. Test không được đọc ở bước này. Bật cờ và chạy cell, sau đó xem kết quả và dừng trước mục 13.

In [8]:
RUN_P1_RANDOM_FOREST_VALIDATION = True
if not RUN_P1_RANDOM_FOREST_VALIDATION:
    raise RuntimeError('Chưa duyệt chạy Random Forest trên Validation.')
import importlib
from pathlib import Path
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import validation_models
importlib.reload(validation_models)
import p1_random_forest_workflow
importlib.reload(p1_random_forest_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_random_forest_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
p1_random_forest_workflow.prepare_validation_cache(p1_validation['run_dir'])
random_forest_validation_results = p1_random_forest_workflow.compare_validation(p1_validation['run_dir'])
display(random_forest_validation_results)


Peak RAM ước tính: 1.87 GiB; RAM khả dụng: 3.82 GiB.
Using validated training cache: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\training_cache
Đang fit và đánh giá Validation: max_depth=12, min_samples_leaf=20
Peak RAM ước tính: 1.87 GiB; RAM khả dụng: 3.80 GiB.
Đang fit và đánh giá Validation: max_depth=16, min_samples_leaf=20
Peak RAM ước tính: 1.87 GiB; RAM khả dụng: 5.70 GiB.
Đang fit và đánh giá Validation: max_depth=20, min_samples_leaf=20
Peak RAM ước tính: 1.87 GiB; RAM khả dụng: 5.70 GiB.
Đang fit và đánh giá Validation: max_depth=20, min_samples_leaf=10
Peak RAM ước tính: 1.87 GiB; RAM khả dụng: 5.95 GiB.
Đang fit và đánh giá Validation: max_depth=20, min_samples_leaf=50
Peak RAM ước tính: 1.87 GiB; RAM khả dụng: 6.15 GiB.
Cấu hình được chọn theo Macro-F1: {'criterion': 'gini', 'max_leaf_nodes': 10000, 'n_estimators': 50, 'max_features': 'sqrt', 'bootstrap': True, 'max_samples': 0.25, 'n_jobs':

,max_depth,min_samples_leaf,max_leaf_nodes,fit_seconds,max_tree_depth_observed,total_leaf_count,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,12,20,10000,103.175571,12,10068,360226,300537,165,1185,58339,0.996252,0.997180,0.980092,0.988562,0.993161,0.000549,0.019908
1,16,20,10000,106.919089,16,16289,360226,300524,178,627,58897,0.997765,0.996987,0.989466,0.993212,0.995937,0.000592,0.010534
2,20,20,10000,114.700147,20,20000,360226,300516,186,577,58947,0.997882,0.996855,0.990306,0.993570,0.996151,0.000619,0.009694
3,20,10,10000,116.995818,20,25696,360226,300536,166,481,59043,0.998204,0.997196,0.991919,0.994551,0.996738,0.000552,0.008081
4,20,50,10000,108.404057,20,14224,360226,300453,249,731,58793,0.997279,0.995783,0.987719,0.991735,0.995053,0.000828,0.012281


## 13. Fit Random Forest đã chọn trên toàn Train

Chỉ bật sau khi đã xem và duyệt kết quả mục 12. Fit cấu hình đã chọn trên toàn Train gốc, lưu model và thông tin huấn luyện; không đọc Test hoặc ghi đè artifact.

In [9]:
RUN_P1_RANDOM_FOREST_FINAL = True
if not RUN_P1_RANDOM_FOREST_FINAL:
    raise RuntimeError('Chưa duyệt fit Random Forest trên toàn Train.')
from pathlib import Path
import sys
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p1_random_forest_workflow
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_random_forest_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
random_forest_model_path = p1_random_forest_workflow.fit_final(p1_validation['run_dir'])
print(random_forest_model_path)


Peak RAM ước tính: 2.19 GiB; RAM khả dụng: 5.87 GiB.
Peak RAM ước tính: 2.19 GiB; RAM khả dụng: 5.70 GiB.
Đã lưu Random Forest: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\random_forest_selected_model.joblib
D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\random_forest_selected_model.joblib


## 14. Đánh giá Random Forest trên Test P1

Chỉ chạy sau khi model cuối đã fit và được duyệt. Đánh giá Accuracy, Precision/Recall Attack, Macro-F1, FPR/FNR và TN/FP/FN/TP; không dùng Test để chọn hoặc điều chỉnh cấu hình. Bảng output đã được khôi phục từ metrics JSON, không phải output của một lần dự đoán mới.

In [ ]:
RUN_P1_RANDOM_FOREST_TEST = True
if not RUN_P1_RANDOM_FOREST_TEST:
    raise RuntimeError('Chưa duyệt đánh giá Random Forest trên Test P1.')
from pathlib import Path
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p1_random_forest_workflow
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_random_forest_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
random_forest_test_results = p1_random_forest_workflow.evaluate_test(p1_validation['run_dir'])
display(random_forest_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,771979,643875,347,951,126806,0.998319,0.997271,0.992556,0.994908,0.996951,0.000539,0.007444


## 15. XGBoost — so sánh trên Validation

Dùng cùng 78 feature, Train nội bộ và Validation P1; không dùng scaler. Thử tuần tự các bộ `(max_depth, n_estimators, min_child_weight)`: (4, 150, 1), (6, 150, 1), (8, 150, 1), (6, 250, 1), (6, 150, 5). Cố định `learning_rate=0.1`, `tree_method='hist'`, `max_bin=128`, `subsample=0.8`, `colsample_bytree=0.8`, `reg_lambda=1`, `scale_pos_weight=1`, `random_state=42`, `n_jobs=4`, CPU; không early stopping. Chọn Macro-F1 cao nhất; hòa chính xác ưu tiên depth nhỏ, ít cây, rồi child weight lớn. Trước mỗi lần fit kiểm tra tối thiểu 4 GiB RAM khả dụng và 3 GiB đĩa trống; đây không phải giới hạn cứng cho peak RAM. Test không được đọc ở bước này. Bật cờ và chạy riêng cell này; xem bảng rồi dừng trước mục 16.

In [3]:
RUN_P1_XGBOOST_VALIDATION = True
if not RUN_P1_XGBOOST_VALIDATION:
    raise RuntimeError('Chưa duyệt chạy XGBoost trên Validation.')
import importlib
from pathlib import Path
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import validation_models
importlib.reload(validation_models)
import p1_xgboost_workflow
importlib.reload(p1_xgboost_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_xgboost_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
p1_xgboost_workflow.prepare_validation_cache(p1_validation['run_dir'])
xgboost_validation_results = p1_xgboost_workflow.compare_validation(p1_validation['run_dir'])
display(xgboost_validation_results)


RAM khả dụng: 5.74 GiB; đĩa trống: 67.12 GiB. Đây không phải giới hạn peak RAM thực tế.
Using validated training cache: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\training_cache
Đang fit và đánh giá Validation: {'max_depth': 4, 'n_estimators': 150, 'min_child_weight': 1}
RAM khả dụng: 5.73 GiB; đĩa trống: 67.12 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit và đánh giá Validation: {'max_depth': 6, 'n_estimators': 150, 'min_child_weight': 1}
RAM khả dụng: 5.84 GiB; đĩa trống: 67.12 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit và đánh giá Validation: {'max_depth': 8, 'n_estimators': 150, 'min_child_weight': 1}
RAM khả dụng: 5.83 GiB; đĩa trống: 67.12 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit và đánh giá Validation: {'max_depth': 6, 'n_estimators': 250, 'min_child_weight': 1}
RAM khả dụng: 5.95 GiB; đĩa trống: 67.12 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit và

,max_depth,n_estimators,min_child_weight,fit_seconds,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,4,150,1,16.450705,360226,300502,200,302,59222,0.998606,0.996634,0.994926,0.995780,0.997473,0.000665,0.005074
1,6,150,1,19.772622,360226,300508,194,132,59392,0.999095,0.996744,0.997782,0.997263,0.998360,0.000645,0.002218
2,8,150,1,20.666040,360226,300513,189,114,59410,0.999159,0.996829,0.998085,0.997456,0.998476,0.000629,0.001915
3,6,250,1,29.635811,360226,300517,185,112,59412,0.999176,0.996896,0.998118,0.997507,0.998506,0.000615,0.001882
4,6,150,5,18.616971,360226,300502,200,137,59387,0.999064,0.996644,0.997698,0.997171,0.998305,0.000665,0.002302


## 16. Fit XGBoost đã chọn trên toàn Train

Chỉ bật sau khi đã xem và duyệt kết quả mục 15. Fit cấu hình đã chọn trên toàn Train gốc, lưu model và thông tin huấn luyện; không đọc Test hoặc ghi đè artifact.

In [4]:
RUN_P1_XGBOOST_FINAL = True
if not RUN_P1_XGBOOST_FINAL:
    raise RuntimeError('Chưa duyệt fit XGBoost trên toàn Train.')
from pathlib import Path
import sys
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p1_xgboost_workflow
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_xgboost_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
xgboost_model_path = p1_xgboost_workflow.fit_final(p1_validation['run_dir'])
print(xgboost_model_path)


RAM khả dụng: 5.70 GiB; đĩa trống: 67.12 GiB. Đây không phải giới hạn peak RAM thực tế.
RAM khả dụng: 5.32 GiB; đĩa trống: 66.59 GiB. Đây không phải giới hạn peak RAM thực tế.
Đã lưu XGBoost: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\xgboost_selected_model.joblib
D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\xgboost_selected_model.joblib


## 17. Đánh giá XGBoost trên Test P1

Chỉ chạy sau khi model cuối đã fit và được duyệt. Đánh giá Accuracy, Precision/Recall Attack, Macro-F1, FPR/FNR và TN/FP/FN/TP; không dùng Test để chọn hoặc điều chỉnh cấu hình.

In [5]:
RUN_P1_XGBOOST_TEST = True
if not RUN_P1_XGBOOST_TEST:
    raise RuntimeError('Chưa duyệt đánh giá XGBoost trên Test P1.')
from pathlib import Path
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p1_xgboost_workflow
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_xgboost_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
xgboost_test_results = p1_xgboost_workflow.evaluate_test(p1_validation['run_dir'])
display(xgboost_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,771979,643824,398,212,127545,0.99921,0.996889,0.998341,0.997614,0.99857,0.000618,0.001659


## 18. MLP — chọn cấu hình trên Validation

Dùng `Pipeline(StandardScaler, MLPClassifier)` để scaler chỉ fit trên Train nội bộ. So sánh một lớp ẩn `(32,)`, `(64,)`, `(128,)`; cố định ReLU, Adam, `alpha=0.0001`, `batch_size=1024`, `learning_rate_init=0.001`, `max_iter=100`, `early_stopping=False`, `random_state=42`. Chọn theo Macro-F1; hòa chính xác thì chọn mạng ít neuron hơn. Theo dõi thời gian, số vòng lặp và cảnh báo chưa hội tụ. Preflight RAM/đĩa trước từng lần fit; ngưỡng không bảo đảm peak RAM thực tế. Cả ba cấu hình ở lượt này có cảnh báo hội tụ; mục 18.1 khảo sát lại trước khi chọn cấu hình fit toàn Train.

In [1]:
RUN_P1_MLP_VALIDATION = True
if not RUN_P1_MLP_VALIDATION:
    raise RuntimeError('Chưa duyệt chạy MLP trên Validation.')
import importlib
from pathlib import Path
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import validation_models
importlib.reload(validation_models)
import p1_mlp_workflow
importlib.reload(p1_mlp_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_mlp_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
p1_mlp_workflow.prepare_validation_cache(p1_validation['run_dir'])
mlp_validation_results = p1_mlp_workflow.compare_validation(p1_validation['run_dir'])
display(mlp_validation_results)


Đã nạp lại Validation run: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2
Using validated training cache: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\training_cache
Đang fit và đánh giá Validation: hidden_layer_sizes=(32,)
Peak RAM ước tính: 1.80 GiB; RAM khả dụng: 6.83 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đang fit và đánh giá Validation: hidden_layer_sizes=(64,)
Peak RAM ước tính: 1.80 GiB; RAM khả dụng: 7.49 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đang fit và đánh giá Validation: hidden_layer_sizes=(128,)
Peak RAM ước tính: 1.80 GiB; RAM khả dụng: 7.75 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Cấu hình MLP được chọn theo Macro-F1: {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 1024, 'early_stopping': False, 'learning_rate_init': 0.001,

,hidden_units,fit_seconds,n_iter,final_loss,convergence_warnings,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,32,197.795610,100,0.016975,[Stochastic Optimizer: Maximum iterations (100...,360226,300257,445,930,58594,0.996183,0.992463,0.984376,0.988403,0.993059,0.001480,0.015624
1,64,262.209537,100,0.014069,[Stochastic Optimizer: Maximum iterations (100...,360226,300153,549,645,58879,0.996685,0.990762,0.989164,0.989962,0.993989,0.001826,0.010836
2,128,384.612586,100,0.012958,[Stochastic Optimizer: Maximum iterations (100...,360226,300234,468,843,58681,0.996361,0.992088,0.985838,0.988953,0.993387,0.001556,0.014162


### 18.1. Khảo sát hội tụ MLP trên Validation

Fit lại từ đầu ba cấu hình 32/64/128 neuron với tối đa 200 vòng; giữ nguyên các tham số khác và dữ liệu Train nội bộ/Validation. So sánh với kết quả tối đa 100 vòng đã lưu; hiển thị thời gian, số vòng, loss, cảnh báo và metrics. Ghi artifact riêng, không dùng Test hoặc ghi đè kết quả 100 vòng. Sau khi xem kết quả, người dùng chọn 128 neuron/200 vòng cho mục 19.

In [3]:
RUN_P1_MLP_CONVERGENCE_200 = True
if not RUN_P1_MLP_CONVERGENCE_200:
    raise RuntimeError('Chưa duyệt khảo sát MLP với tối đa 200 vòng.')
import importlib
from pathlib import Path
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import validation_models
importlib.reload(validation_models)
import p1_mlp_workflow
importlib.reload(p1_mlp_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_mlp_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
mlp_convergence_results = p1_mlp_workflow.compare_convergence_200(p1_validation['run_dir'])
display(mlp_convergence_results)


Using validated training cache: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\training_cache
Đang fit và đánh giá Validation: hidden_layer_sizes=(32,)
Peak RAM ước tính: 1.80 GiB; RAM khả dụng: 5.29 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đang fit và đánh giá Validation: hidden_layer_sizes=(64,)
Peak RAM ước tính: 1.80 GiB; RAM khả dụng: 5.33 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đang fit và đánh giá Validation: hidden_layer_sizes=(128,)
Peak RAM ước tính: 1.80 GiB; RAM khả dụng: 5.70 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Cấu hình MLP được chọn theo Macro-F1: {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 1024, 'early_stopping': False, 'learning_rate_init': 0.001, 'max_iter': 200, 'random_state': 42, 'solver': 'adam', 'hidden_layer_sizes': [128]}
Khảo sát 200 vòng không thay đổi lựa chọn hiện dùng ở mục 19; cần du

,hidden_units,fit_seconds,n_iter,final_loss,convergence_warnings,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr,max_iter
0,32,197.795610,100,0.016975,[Stochastic Optimizer: Maximum iterations (100...,360226,300257,445,930,58594,0.996183,0.992463,0.984376,0.988403,0.993059,0.001480,0.015624,100
1,64,262.209537,100,0.014069,[Stochastic Optimizer: Maximum iterations (100...,360226,300153,549,645,58879,0.996685,0.990762,0.989164,0.989962,0.993989,0.001826,0.010836,100
2,128,384.612586,100,0.012958,[Stochastic Optimizer: Maximum iterations (100...,360226,300234,468,843,58681,0.996361,0.992088,0.985838,0.988953,0.993387,0.001556,0.014162,100
3,32,343.971428,172,0.012735,[],360226,300379,323,789,58735,0.996913,0.994531,0.986745,0.990623,0.994387,0.001074,0.013255,200
4,64,362.493820,134,0.012793,[],360226,300245,457,640,58884,0.996955,0.992299,0.989248,0.990771,0.994474,0.001520,0.010752,200
5,128,420.291460,103,0.012710,[],360226,300420,282,734,58790,0.997180,0.995226,0.987669,0.991433,0.994872,0.000938,0.012331,200


## 19. Fit MLP đã chọn trên toàn Train

Tạo Pipeline mới với cấu hình 128 neuron, tối đa 200 vòng đã chọn từ mục 18.1; fit scaler và MLP trên toàn Train, lưu pipeline cùng summary. Cell không đọc Test hoặc ghi đè artifact. Chỉ chạy sau khi đã duyệt fit toàn Train.

In [4]:
RUN_P1_MLP_FINAL = True
if not RUN_P1_MLP_FINAL:
    raise RuntimeError('Chưa duyệt fit MLP trên toàn Train.')
from pathlib import Path
import importlib
import sys
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p1_mlp_workflow
importlib.reload(p1_mlp_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_mlp_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
mlp_model_path = p1_mlp_workflow.fit_final(p1_validation['run_dir'])
print(mlp_model_path)


Peak RAM ước tính: 2.22 GiB; RAM khả dụng: 5.38 GiB; đĩa trống: 67.12 GiB. Ước tính không bảo đảm peak RAM thực tế.
Peak RAM ước tính: 2.22 GiB; RAM khả dụng: 5.27 GiB; đĩa trống: 66.59 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đã lưu MLP pipeline: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\mlp_selected_pipeline.joblib
D:\HUST\GR2\NIDS-ML\Dataset\Processed\P1_Random_Group_Aware\20261114_024012_45f47887\validation_runs\20261114_030352_2ac39af2\mlp_selected_pipeline.joblib


## 20. Đánh giá MLP trên Test P1

Chỉ chạy sau khi model cuối đã fit và được duyệt. Pipeline dùng scaler đã fit trên toàn Train để transform Test; không fit lại scaler hoặc điều chỉnh MLP trên Test. Test P1 đã được xem trong đánh giá các mô hình P1 trước đó.

In [5]:
RUN_P1_MLP_TEST = True
if not RUN_P1_MLP_TEST:
    raise RuntimeError('Chưa duyệt đánh giá MLP trên Test P1.')
from pathlib import Path
import importlib
import sys
from IPython.display import display
project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'validation_models.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục project NIDS-ML.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p1_mlp_workflow
importlib.reload(p1_mlp_workflow)
if 'p1_validation' not in globals():
    p1_validation = {'run_dir': p1_mlp_workflow.current_validation_run_dir(globals().get('p1_source_run'))}
    print(f"Đã nạp lại Validation run: {p1_validation['run_dir']}")
mlp_test_results = p1_mlp_workflow.evaluate_test(p1_validation['run_dir'])
display(mlp_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,771979,643427,795,1552,126205,0.99696,0.99374,0.987852,0.990787,0.994483,0.001234,0.012148
